# What Is the Best Way to Reduce Fatigue in Long COVID?

## Abstract

We analysed self-reported treatment outcomes from Long COVID patients who mention fatigue,
drawing on a 1-month snapshot of r/covidlonghaulers stored in `polina_onemonth.db`.
Using user-level sentiment aggregation (one data point per person per drug),
we identified the treatments most frequently discussed by fatigue sufferers,
tested whether their positive-outcome rates exceed chance (binomial test),
compared top treatments head-to-head (Kruskal-Wallis with post-hoc),
and present tiered recommendations by evidence strength.
Generic category labels ("supplements", "medication") and causal-context
treatments are filtered from recommendation analyses.
All results reflect community reporting patterns, not clinical-trial evidence.

## 1. Setup

Import libraries, connect to the database, and configure plotting defaults.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sys
import sqlite3
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats as sp_stats
from pathlib import Path
from IPython.display import display, HTML, Markdown

# Project root (works whether notebook is run from notebooks/ or repo root)
_here = Path('.').resolve()
PROJECT_ROOT = _here.parent if (_here / 'app').exists() is False and (_here.parent / 'app').exists() else _here
sys.path.insert(0, str(PROJECT_ROOT))

from app.analysis.stats import (
    get_user_sentiment,
    run_binomial_test,
    summarize_drug,
    run_comparison,
    run_kruskal_wallis,
    REPORTING_BIAS_DISCLAIMER,
)

DB_PATH = PROJECT_ROOT / 'polina_onemonth.db'
conn = sqlite3.connect(str(DB_PATH))

# Sentiment encoding
SENTIMENT_SCORE = {"positive": 1.0, "mixed": 0.5, "weak": 0.25, "neutral": 0.0, "negative": -1.0}

# Generic treatment names to filter from recommendations
GENERIC_NAMES = {
    'supplements', 'supplement', 'medication', 'medications', 'treatment',
    'treatments', 'therapy', 'vitamins', 'drugs', 'drug', 'medicine',
    'prescription', 'otc', 'over the counter', 'herbal', 'remedy',
    'remedies', 'nootropics', 'nootropic', 'antidepressants', 'antidepressant',
    'stimulants', 'stimulant', 'painkillers', 'painkiller', 'antihistamines',
    'anti-inflammatory', 'anti-inflammatories',
}

# Plotting defaults
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11
COLORS = {"positive": "#2ecc71", "mixed": "#d5d8dc", "negative": "#e74c3c"}

## 2. Data Exploration

Before analysing fatigue treatments, we inspect the database to understand its
size, date range, and the prevalence of fatigue-related conditions.

In [ ]:
# ── Table sizes ────────────────────────────────────────────────────────────
table_rows = {}
for table in ["users", "posts", "treatment", "treatment_reports",
              "user_profiles", "conditions"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    table_rows[table] = n

overview = pd.DataFrame([
    {"Table": k, "Rows": f"{v:,}"} for k, v in table_rows.items()
])
display(HTML(overview.to_html(index=False)))

# ── Date range ─────────────────────────────────────────────────────────────
lo, hi = conn.execute("SELECT MIN(post_date), MAX(post_date) FROM posts").fetchone()

# Handle both epoch-int and string date formats
try:
    date_lo = datetime.datetime.fromtimestamp(int(lo)).date()
    date_hi = datetime.datetime.fromtimestamp(int(hi)).date()
except (ValueError, TypeError, OSError):
    date_lo = pd.to_datetime(str(lo)).date()
    date_hi = pd.to_datetime(str(hi)).date()

months = max(1, round((date_hi - date_lo).days / 30.44))
display(HTML(
    f"<b>Data covers:</b> {date_lo} to {date_hi} ({months} month{'s' if months != 1 else ''})"
))

### Fatigue prevalence

We identify users who have a condition containing the word "fatigue" (case-insensitive),
then count how many of those users also have treatment reports.

In [ ]:
# ── Fatigue users ─────────────────────────────────────────────────────────
fatigue_user_ids = pd.read_sql("""
    SELECT DISTINCT user_id
    FROM conditions
    WHERE LOWER(condition_name) LIKE '%fatigu%'
       OR LOWER(condition_name) LIKE '%exhausti%'
       OR LOWER(condition_name) LIKE '%tired%'
       OR LOWER(condition_name) LIKE '%low energy%'
""", conn)["user_id"]

n_fatigue_users = len(fatigue_user_ids)
total_users = table_rows["users"]

# How many of these have treatment reports?
fatigue_with_tx = pd.read_sql(f"""
    SELECT COUNT(DISTINCT tr.user_id) AS n
    FROM treatment_reports tr
    WHERE tr.user_id IN (
        SELECT DISTINCT user_id FROM conditions
        WHERE LOWER(condition_name) LIKE '%fatigu%'
           OR LOWER(condition_name) LIKE '%exhausti%'
           OR LOWER(condition_name) LIKE '%tired%'
           OR LOWER(condition_name) LIKE '%low energy%'
    )
""", conn).iloc[0, 0]

display(HTML(
    f"<b>Fatigue cohort:</b> {n_fatigue_users:,} users mention fatigue-related conditions "
    f"({n_fatigue_users / total_users * 100:.1f}% of all users). "
    f"Of these, <b>{fatigue_with_tx:,}</b> have at least one treatment report."
))

### Dataset overview: condition prevalence and sentiment distribution

Two-panel figure showing (left) the most common conditions in the dataset, highlighting
fatigue-related ones, and (right) the overall sentiment distribution across all treatment reports.

In [ ]:
# ── Top conditions ─────────────────────────────────────────────────────────
top_conditions = pd.read_sql("""
    SELECT condition_name, COUNT(DISTINCT user_id) AS n_users
    FROM conditions
    GROUP BY condition_name
    ORDER BY n_users DESC
    LIMIT 15
""", conn)

# ── Sentiment distribution ─────────────────────────────────────────────────
sent_dist = pd.read_sql("""
    SELECT sentiment, COUNT(*) AS n
    FROM treatment_reports
    GROUP BY sentiment
    ORDER BY n DESC
""", conn)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: condition bar chart
fatigue_keywords = ['fatigu', 'exhaust', 'tired', 'low energy']
is_fatigue = top_conditions['condition_name'].apply(
    lambda c: any(kw in c.lower() for kw in fatigue_keywords)
)
bar_colors = ['#e67e22' if f else '#3498db' for f in is_fatigue]

axes[0].barh(
    top_conditions['condition_name'][::-1],
    top_conditions['n_users'][::-1],
    color=bar_colors[::-1],
    edgecolor='white', linewidth=0.5
)
axes[0].set_xlabel('Number of users')
axes[0].set_title('Top 15 Conditions\n(orange = fatigue-related)')

# Right: sentiment pie
sent_colors = {
    'positive': '#2ecc71', 'mixed': '#f39c12', 'weak': '#d5d8dc',
    'neutral': '#95a5a6', 'negative': '#e74c3c'
}
pie_colors = [sent_colors.get(s, '#bdc3c7') for s in sent_dist['sentiment']]
axes[1].pie(
    sent_dist['n'], labels=sent_dist['sentiment'],
    autopct='%1.1f%%', colors=pie_colors, startangle=90
)
axes[1].set_title('Overall Sentiment Distribution\n(all treatment reports)')

plt.tight_layout()
plt.show()

The left panel shows the most commonly reported conditions in this r/covidlonghaulers snapshot.
Fatigue-related conditions are highlighted in orange. The right panel shows the overall
sentiment distribution across all treatment reports -- this is the baseline we compare against
when testing whether individual treatments perform better than chance.

## 3. Treatment Landscape for Fatigue Patients

Which treatments are fatigue patients discussing, and what are the reported outcomes?
We build a user-level summary (one data point per person per drug) for all treatments
reported by users who mention fatigue, then filter out generic category names that
a patient cannot act on.

In [ ]:
# ── All treatment reports for fatigue users ────────────────────────────────
df_all = pd.read_sql("""
    SELECT tr.user_id, t.canonical_name AS drug, tr.sentiment, tr.signal_strength
    FROM treatment_reports tr
    JOIN treatment t ON t.id = tr.drug_id
    WHERE tr.user_id IN (
        SELECT DISTINCT user_id FROM conditions
        WHERE LOWER(condition_name) LIKE '%fatigu%'
           OR LOWER(condition_name) LIKE '%exhausti%'
           OR LOWER(condition_name) LIKE '%tired%'
           OR LOWER(condition_name) LIKE '%low energy%'
    )
""", conn)

df_all['score'] = df_all['sentiment'].map(SENTIMENT_SCORE)

# User-level aggregation
user_drug = df_all.groupby(['drug', 'user_id']).agg(
    avg_score=('score', 'mean'),
    n_reports=('score', 'count')
).reset_index()

user_drug['outcome'] = user_drug['avg_score'].apply(
    lambda x: 'positive' if x > 0.7 else ('negative' if x < -0.7 else 'mixed/neutral')
)

# Per-drug summary
drug_summary = user_drug.groupby('drug').agg(
    n_users=('user_id', 'count'),
    pct_positive=('outcome', lambda x: (x == 'positive').mean() * 100),
    pct_negative=('outcome', lambda x: (x == 'negative').mean() * 100),
    avg_score=('avg_score', 'mean')
).reset_index()

drug_summary['pct_mixed'] = 100 - drug_summary['pct_positive'] - drug_summary['pct_negative']

# Filter out generic names
drug_summary['is_generic'] = drug_summary['drug'].str.lower().isin(GENERIC_NAMES)
drug_filtered = drug_summary[~drug_summary['is_generic']].copy()

n_generic_removed = drug_summary['is_generic'].sum()

display(HTML(
    f"<b>Treatment landscape:</b> {len(df_all):,} reports across "
    f"{drug_summary['drug'].nunique()} treatments from {user_drug['user_id'].nunique()} fatigue users. "
    f"Filtered {n_generic_removed} generic category names."
))

# Show top 20 by user count
top20 = drug_filtered.sort_values('n_users', ascending=False).head(20)[[
    'drug', 'n_users', 'pct_positive', 'pct_negative', 'pct_mixed', 'avg_score'
]].copy()
top20.columns = ['Treatment', 'Users', '% Positive', '% Negative', '% Mixed', 'Avg Score']
top20 = top20.round(1)
display(HTML(top20.to_html(index=False)))

### Diverging bar chart: treatment outcomes for fatigue patients

The chart below shows the proportion of positive, mixed/neutral, and negative
user-level outcomes for the most-discussed treatments among fatigue patients.
Only treatments with 3 or more unique reporters are shown.

In [ ]:
# Diverging bar chart — top treatments (n >= 3)
chart_data = drug_filtered[drug_filtered['n_users'] >= 3].sort_values(
    'pct_positive', ascending=True
).tail(20)

n_bars = len(chart_data)
fig, ax = plt.subplots(figsize=(12, max(5, n_bars * 0.45)))

y = np.arange(n_bars)
pct_pos = chart_data['pct_positive'].values
pct_neg = chart_data['pct_negative'].values
pct_mix = chart_data['pct_mixed'].values
drugs = chart_data['drug'].values
ns = chart_data['n_users'].values

# Correct stacking: mixed INNERMOST, negative OUTERMOST
ax.barh(y, -pct_mix, height=0.6, color=COLORS['mixed'],
        label='Mixed/Neutral', edgecolor='white', linewidth=0.5)
ax.barh(y, -pct_neg, height=0.6, left=-pct_mix, color=COLORS['negative'],
        label='Negative', edgecolor='white', linewidth=0.5)
ax.barh(y, pct_pos, height=0.6, color=COLORS['positive'],
        label='Positive', edgecolor='white', linewidth=0.5)

ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_yticks(y)
ax.set_yticklabels([d.title() for d in drugs], fontsize=10)

# n= labels
for i in range(n_bars):
    ax.text(pct_pos[i] + 2, i, f'n={ns[i]}', va='center', fontsize=8, color='#555')

max_extent = max(max(pct_neg + pct_mix), max(pct_pos)) + 10
ax.set_xlim(-max_extent, max_extent + 10)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{abs(x):.0f}%'))
ax.set_xlabel('\u2190 Negative          Positive \u2192')
ax.set_title('Treatment Outcomes Among Fatigue Patients\n(user-level, r/covidlonghaulers)', pad=12)

ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.08), ncol=3, frameon=False, fontsize=10)
fig.subplots_adjust(bottom=0.15)

for spine in ['top', 'right', 'left']:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

The diverging bar chart ranks treatments by their positive-outcome rate among fatigue
patients. Treatments at the top of the chart have the highest proportion of users
reporting positive experiences. The `n=` labels indicate how many unique users discussed
each treatment -- larger samples give more reliable estimates.

**Key observations:**
- Treatments near the top (highest positive %) are the strongest candidates for fatigue relief.
- Any treatment with a large red (negative) bar warrants caution.
- Small-n treatments should be interpreted as preliminary signals, not confirmed recommendations.

## 4. Statistical Testing: Do Any Treatments Beat Chance?

For each treatment with 5 or more unique fatigue reporters, we run a one-sample
binomial test asking: *Is the positive-outcome rate significantly different from 50%?*
The 50% baseline represents chance -- if a treatment is no better than a coin flip,
we would expect about half of users to report positive outcomes.

**What this means for patients:** A treatment that "beats chance" has a positive-outcome
rate high enough that it is unlikely to be a fluke. This is a minimum bar -- it does not
prove the treatment works, but it means the signal in the data is worth paying attention to.

In [ ]:
# Binomial tests for treatments with n >= 5 among fatigue users
eligible_drugs = drug_filtered[drug_filtered['n_users'] >= 5].sort_values(
    'n_users', ascending=False
)

binom_rows = []
for _, row in eligible_drugs.iterrows():
    drug_name = row['drug']
    df_drug = get_user_sentiment(conn, drug_name, condition='%fatigu%')
    if df_drug.empty:
        # Fallback: use our pre-computed user-drug data
        sub = user_drug[user_drug['drug'] == drug_name]
        n = len(sub)
        if n < 5:
            continue
        n_pos = (sub['outcome'] == 'positive').sum()
        from scipy.stats import binomtest
        result = binomtest(n_pos, n, p=0.5, alternative='two-sided')
        pct_pos = round(n_pos / n * 100, 1)
        binom_rows.append({
            'Treatment': drug_name,
            'Users': n,
            '% Positive': pct_pos,
            'p-value': round(result.pvalue, 4),
            'Significant': 'Yes' if result.pvalue < 0.05 else '',
            'Mean Score': round(sub['avg_score'].mean(), 2),
        })
    else:
        binom_result = run_binomial_test(df_drug)
        if binom_result is None:
            continue
        summary = summarize_drug(df_drug, drug_name)
        binom_rows.append({
            'Treatment': drug_name,
            'Users': binom_result.n_users,
            '% Positive': round(binom_result.observed_rate * 100, 1),
            'p-value': binom_result.p_value,
            'Significant': 'Yes' if binom_result.significant else '',
            'Mean Score': summary.mean_sentiment if summary else 0,
        })

binom_df = pd.DataFrame(binom_rows).sort_values('Users', ascending=False)

display(HTML('<b>Binomial test: positive rate vs 50% baseline</b> (treatments with 5+ fatigue reporters)'))
display(HTML(binom_df.head(25).to_html(index=False)))

### Forest plot: mean sentiment with 95% confidence intervals

A forest plot shows the mean sentiment score (dot) and its 95% confidence interval
(horizontal bar) for each treatment. Treatments whose CI does not cross zero have
a statistically meaningful positive or negative signal. Wider CIs indicate less
certainty (usually due to smaller sample size).

In [ ]:
# Forest plot for all eligible treatments
forest_drugs = drug_filtered[drug_filtered['n_users'] >= 5].sort_values(
    'avg_score', ascending=True
)

forest_data = []
for _, row in forest_drugs.iterrows():
    drug_name = row['drug']
    scores = user_drug[user_drug['drug'] == drug_name]['avg_score'].values
    n = len(scores)
    if n < 5:
        continue
    mean = scores.mean()
    se = sp_stats.sem(scores)
    ci = se * sp_stats.t.ppf(0.975, n - 1) if n >= 2 else 0
    forest_data.append({'drug': drug_name, 'mean': mean, 'ci': ci, 'n': n})

if forest_data:
    fd = pd.DataFrame(forest_data).sort_values('mean', ascending=True)
    n_items = len(fd)
    fig, ax = plt.subplots(figsize=(10, max(5, n_items * 0.4)))

    y = np.arange(n_items)
    means = fd['mean'].values
    cis = fd['ci'].values
    colors_fp = ['#2ecc71' if m > 0.3 else '#e74c3c' if m < -0.3 else '#95a5a6'
                 for m in means]

    ax.errorbar(means, y, xerr=cis, fmt='none', ecolor='#555',
                elinewidth=1.2, capsize=3, zorder=1)
    ax.scatter(means, y, c=colors_fp, s=70, zorder=2,
               edgecolors='white', linewidth=0.5)
    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)

    ax.set_yticks(y)
    ax.set_yticklabels([d.title() for d in fd['drug'].values], fontsize=10)
    ax.set_xlabel('Mean sentiment score (95% CI)\n-1 = negative, 0 = neutral, +1 = positive')
    ax.set_title('Forest Plot: Treatment Sentiment Among Fatigue Patients', pad=12)

    # n labels
    for i, row_fd in enumerate(fd.itertuples()):
        ax.text(1.15, i, f'n={row_fd.n}', va='center', fontsize=8, color='gray')

    ax.set_xlim(-1.3, 1.3)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)

    plt.tight_layout()
    plt.show()

**How to read this chart:** Each dot is the average sentiment for a treatment. The horizontal
bar is the 95% confidence interval -- if it does not cross the dashed zero line, we can
be reasonably confident that the treatment's community signal is genuinely positive (or negative).

**What this means:** Treatments with green dots to the right of zero and narrow CIs have
the most reliable positive signals. Treatments with wide CIs spanning zero need more data
before we can draw conclusions.

## 5. Head-to-Head Comparison: Top Fatigue Treatments

Do the top treatments differ significantly from one another? We use a Kruskal-Wallis
test (the non-parametric equivalent of one-way ANOVA) to check whether at least one
treatment group has a different sentiment distribution. If significant, post-hoc pairwise
comparisons identify which specific treatments differ.

**What this means for patients:** This section answers the question "Is Treatment A actually
better than Treatment B, or are the differences just noise?" Only statistically significant
differences should influence treatment decisions.

In [ ]:
# Kruskal-Wallis across top treatments with n >= 5
kw_drug_names = drug_filtered[drug_filtered['n_users'] >= 5].sort_values(
    'n_users', ascending=False
).head(8)['drug'].tolist()

kw_groups = []
kw_labels = []
for d in kw_drug_names:
    scores = user_drug[user_drug['drug'] == d]['avg_score'].values
    if len(scores) >= 5:
        kw_groups.append(scores)
        kw_labels.append(d)

if len(kw_groups) >= 3:
    h_stat, kw_p = sp_stats.kruskal(*kw_groups)
    display(HTML(
        f"<b>Kruskal-Wallis test</b> across {len(kw_labels)} treatments: "
        f"H = {h_stat:.2f}, p = {kw_p:.4f}<br>"
        f"<b>{'Significant' if kw_p < 0.05 else 'Not significant'}</b> "
        f"{'\u2014 at least one treatment differs from the others.' if kw_p < 0.05 else '\u2014 no strong evidence that treatments differ.'}"
    ))

    # Grouped bar chart showing outcome distribution
    kw_data = user_drug[user_drug['drug'].isin(kw_labels)].copy()
    counts = (
        kw_data.groupby(['drug', 'outcome'])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=['negative', 'mixed/neutral', 'positive'], fill_value=0)
    )
    # Order by positive count descending
    counts = counts.loc[
        counts['positive'].sort_values(ascending=False).index
    ]

    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(counts))
    w = 0.25
    bar_colors_kw = {
        'negative': '#e74c3c', 'mixed/neutral': '#95a5a6', 'positive': '#2ecc71'
    }
    for j, cat in enumerate(['negative', 'mixed/neutral', 'positive']):
        ax.bar(x + (j - 1) * w, counts[cat], width=w,
               color=bar_colors_kw[cat], label=cat.replace('/', ' / ').title(),
               edgecolor='white')

    ax.set_xticks(x)
    ax.set_xticklabels([d.title() for d in counts.index], rotation=20, ha='right')
    ax.set_ylabel('Number of users')
    ax.set_title(
        f'Outcome Distribution: Top {len(kw_labels)} Fatigue Treatments\n'
        f'(Kruskal-Wallis p = {kw_p:.3f})',
        pad=12
    )
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=3, frameon=False)
    fig.subplots_adjust(bottom=0.25)

    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)

    plt.tight_layout()
    plt.show()
else:
    display(HTML(
        '<b>Fewer than 3 treatments with 5+ users</b> \u2014 '
        'cannot perform Kruskal-Wallis comparison. Presenting descriptive results only.'
    ))

**What this means:** The grouped bar chart shows how many users in each treatment group
reported positive, mixed/neutral, or negative outcomes. Treatments with taller green bars
relative to their red bars are performing well in this community. The Kruskal-Wallis
p-value tells us whether the differences between treatments are statistically meaningful
or could be due to chance.

## 6. Causal-Context Check

In patient communities, some treatments appear in the data primarily because they
*caused or worsened* the condition, not because they were used for recovery.
We check for treatments whose negative sentiment may reflect this pattern
rather than failed fatigue treatment.

In [ ]:
# Identify treatments with very low positive rates that may reflect causal context
# In Long COVID communities, vaccines and certain drugs are often discussed as triggers
causal_candidates = drug_filtered[
    (drug_filtered['n_users'] >= 3) &
    (drug_filtered['pct_positive'] < 30)
].sort_values('pct_positive', ascending=True)

# Known causal-context patterns in Long COVID communities
CAUSAL_KEYWORDS = [
    'vaccine', 'vaccination', 'pfizer', 'moderna', 'astrazeneca',
    'johnson', 'j&j', 'booster', 'jab',
]

causal_flags = []
for _, row in causal_candidates.iterrows():
    drug_lower = row['drug'].lower()
    is_causal = any(kw in drug_lower for kw in CAUSAL_KEYWORDS)
    causal_flags.append({
        'Treatment': row['drug'],
        'Users': row['n_users'],
        '% Positive': round(row['pct_positive'], 1),
        'Likely Causal Context': 'Yes' if is_causal else 'Review',
        'Note': (
            'Negative sentiment likely reflects the trigger/cause of Long COVID, '
            'not treatment failure.'
            if is_causal else
            'Low positive rate -- may reflect genuine poor outcomes or causal context.'
        )
    })

if causal_flags:
    causal_df = pd.DataFrame(causal_flags)
    display(HTML('<b>Causal-context review:</b> treatments with < 30% positive outcomes'))
    display(HTML(causal_df.to_html(index=False)))
    display(HTML(
        '<i>Treatments flagged as "Likely Causal Context" are excluded from '
        'recovery recommendations below. Their negative sentiment reflects why '
        'users are in the Long COVID community, not the drug\'s efficacy for fatigue recovery.</i>'
    ))
else:
    display(HTML('<i>No treatments flagged for causal-context contamination.</i>'))

# Build exclusion set
causal_exclude = set()
for f in causal_flags:
    if f['Likely Causal Context'] == 'Yes':
        causal_exclude.add(f['Treatment'].lower())

### Heatmap: co-occurring conditions among fatigue patients

Understanding which conditions co-occur with fatigue helps contextualise treatment choices.
A treatment that works for fatigue + POTS may differ from one for fatigue + brain fog.

In [ ]:
# Co-occurring conditions for fatigue users
cooccur = pd.read_sql("""
    SELECT c.condition_name, COUNT(DISTINCT c.user_id) AS n_users
    FROM conditions c
    WHERE c.user_id IN (
        SELECT DISTINCT user_id FROM conditions
        WHERE LOWER(condition_name) LIKE '%fatigu%'
           OR LOWER(condition_name) LIKE '%exhausti%'
    )
    AND LOWER(c.condition_name) NOT LIKE '%fatigu%'
    AND LOWER(c.condition_name) NOT LIKE '%exhausti%'
    GROUP BY c.condition_name
    ORDER BY n_users DESC
    LIMIT 12
""", conn)

if not cooccur.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(
        data=cooccur, x='n_users', y='condition_name',
        hue='condition_name', palette='viridis', dodge=False,
        legend=False, ax=ax
    )
    ax.set_xlabel('Number of fatigue users also reporting this condition')
    ax.set_ylabel('')
    ax.set_title('Conditions Co-occurring with Fatigue', pad=10)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    plt.tight_layout()
    plt.show()
else:
    display(HTML('<i>No co-occurring condition data available.</i>'))

The bar chart above shows which other conditions fatigue patients most commonly report.
This comorbidity profile is important because many Long COVID treatments target specific
symptom clusters (e.g., LDN for neuroinflammation, beta-blockers for POTS) rather than
fatigue in isolation.

## 7. Recommendations

Treatments are grouped into three tiers based on evidence strength:

- **Strong evidence:** 30+ unique reporters AND positive rate significantly above 50% (p < 0.05)
- **Moderate evidence:** 15+ reporters OR borderline significance (p < 0.10)
- **Preliminary:** Fewer than 15 reporters, not statistically significant, but positive signal

Generic category names and causal-context treatments are excluded.

In [ ]:
# Build recommendation tiers
rec_rows = []
for brow in binom_rows:
    drug_lower = brow['Treatment'].lower()
    # Skip generic names and causal-context drugs
    if drug_lower in GENERIC_NAMES or drug_lower in causal_exclude:
        continue
    n = brow['Users']
    pct = brow['% Positive']
    p = brow['p-value']
    sig = brow['Significant'] == 'Yes'

    if n >= 30 and sig:
        tier = 'Strong'
    elif n >= 15 or p < 0.10:
        tier = 'Moderate'
    elif pct > 50:
        tier = 'Preliminary'
    else:
        continue  # skip drugs with <= 50% positive

    rec_rows.append({
        'Tier': tier,
        'Treatment': brow['Treatment'],
        'Users': n,
        '% Positive': pct,
        'p-value': p,
        'Mean Score': brow['Mean Score'],
    })

# Also add drugs with n >= 3 but < 5 (not tested, but signal > 50%)
small_n = drug_filtered[
    (drug_filtered['n_users'] >= 3) &
    (drug_filtered['n_users'] < 5) &
    (drug_filtered['pct_positive'] > 50) &
    (~drug_filtered['drug'].str.lower().isin(GENERIC_NAMES)) &
    (~drug_filtered['drug'].str.lower().isin(causal_exclude))
]
for _, row in small_n.iterrows():
    rec_rows.append({
        'Tier': 'Preliminary',
        'Treatment': row['drug'],
        'Users': row['n_users'],
        '% Positive': round(row['pct_positive'], 1),
        'p-value': None,
        'Mean Score': round(row['avg_score'], 2),
    })

rec_df = pd.DataFrame(rec_rows)

# Sort: Strong first, then Moderate, then Preliminary; within tier by % Positive desc
tier_order = {'Strong': 0, 'Moderate': 1, 'Preliminary': 2}
if not rec_df.empty:
    rec_df['_sort'] = rec_df['Tier'].map(tier_order)
    rec_df = rec_df.sort_values(['_sort', '% Positive'], ascending=[True, False]).drop(
        columns='_sort'
    )

# Format for display
tier_colors = {'Strong': '#27ae60', 'Moderate': '#f39c12', 'Preliminary': '#95a5a6'}

# Display by tier
for tier_name in ['Strong', 'Moderate', 'Preliminary']:
    tier_data = rec_df[rec_df['Tier'] == tier_name]
    if tier_data.empty:
        continue
    tier_label = {
        'Strong': 'Strong Evidence (n >= 30, p < 0.05)',
        'Moderate': 'Moderate Evidence (n >= 15 or p < 0.10)',
        'Preliminary': 'Preliminary (signal > 50% positive, needs more data)',
    }[tier_name]
    display(HTML(f'<h4 style="color:{tier_colors[tier_name]}">{tier_label}</h4>'))
    show_cols = ['Treatment', 'Users', '% Positive', 'p-value', 'Mean Score']
    display(HTML(tier_data[show_cols].to_html(index=False, na_rep='--')))

### Visual summary: recommended treatments by tier

A single diverging bar chart showing only the recommended treatments, colour-coded
by evidence tier. This is the key take-away chart for this notebook.

In [ ]:
# Visual summary of recommendations
if not rec_df.empty:
    # Merge with drug_summary for full pct data
    vis = rec_df.merge(
        drug_filtered[['drug', 'pct_positive', 'pct_negative', 'pct_mixed']],
        left_on='Treatment', right_on='drug', how='left', suffixes=('', '_full')
    ).sort_values('pct_positive', ascending=True)

    n_bars = len(vis)
    fig, ax = plt.subplots(figsize=(12, max(5, n_bars * 0.4)))

    y = np.arange(n_bars)
    pct_pos_v = vis['pct_positive'].values
    pct_neg_v = vis['pct_negative'].values
    pct_mix_v = vis['pct_mixed'].values

    # Correct stacking: mixed INNERMOST, negative OUTERMOST
    ax.barh(y, -pct_mix_v, height=0.6, color=COLORS['mixed'],
            label='Mixed/Neutral', edgecolor='white', linewidth=0.5)
    ax.barh(y, -pct_neg_v, height=0.6, left=-pct_mix_v, color=COLORS['negative'],
            label='Negative', edgecolor='white', linewidth=0.5)
    ax.barh(y, pct_pos_v, height=0.6, color=COLORS['positive'],
            label='Positive', edgecolor='white', linewidth=0.5)

    ax.axvline(x=0, color='black', linewidth=0.8)

    # Tier colour on y-axis labels
    tier_label_colors = [tier_colors.get(t, 'black') for t in vis['Tier'].values]
    ax.set_yticks(y)
    labels = [f"{d.title()} (n={n})" for d, n in zip(vis['Treatment'].values, vis['Users'].values)]
    ax.set_yticklabels(labels, fontsize=10)
    for tick_label, color in zip(ax.get_yticklabels(), tier_label_colors):
        tick_label.set_color(color)

    # Tier markers
    for i, tier_val in enumerate(vis['Tier'].values):
        marker = {'Strong': '\u2605', 'Moderate': '\u25CF', 'Preliminary': '\u25CB'}[tier_val]
        ax.text(-max(pct_neg_v + pct_mix_v) - 12, i, marker,
                va='center', ha='center', fontsize=12,
                color=tier_colors.get(tier_val, 'gray'))

    max_extent = max(max(pct_neg_v + pct_mix_v + 5, default=50), max(pct_pos_v + 5, default=50))
    ax.set_xlim(-max_extent - 15, max_extent + 5)
    ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{abs(x):.0f}%'))
    ax.set_xlabel('\u2190 Negative          Positive \u2192')
    ax.set_title(
        'Recommended Treatments for Long COVID Fatigue\n'
        '(\u2605 Strong  \u25CF Moderate  \u25CB Preliminary)',
        pad=12, fontsize=13
    )

    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.08), ncol=3, frameon=False)
    fig.subplots_adjust(bottom=0.15)

    for spine in ['top', 'right', 'left']:
        ax.spines[spine].set_visible(False)

    plt.tight_layout()
    plt.show()
else:
    display(HTML('<i>No treatments met the threshold for recommendation.</i>'))

**How to read this chart:** Treatments are sorted from lowest to highest positive-outcome
rate. Label colours and symbols indicate the evidence tier:
green stars = strong evidence, orange dots = moderate, grey circles = preliminary.

**What this means for patients:** Treatments near the top of the chart with green labels
have the most community evidence behind them. Discuss these with your doctor as potential
options. Treatments with grey labels show promise but have fewer reports -- consider them
as "worth exploring" rather than "proven."

**What this means for researchers:** The tiered system combines sample size and statistical
significance. Strong-tier treatments clear both bars (n >= 30, p < 0.05). Moderate-tier
treatments have reasonable sample sizes or borderline significance. Preliminary treatments
are hypothesis-generating only.

## Summary

In [ ]:
# Build dynamic summary
summary_parts = []
summary_parts.append('### Key Findings\n')
summary_parts.append(
    f'- **{len(df_all):,} treatment reports** from **{user_drug["user_id"].nunique()}** '
    f'fatigue patients across **{drug_filtered["drug"].nunique()}** specific treatments '
    f'(after filtering {n_generic_removed} generic category names).'
)

for tier_name in ['Strong', 'Moderate', 'Preliminary']:
    tier_data = rec_df[rec_df['Tier'] == tier_name] if not rec_df.empty else pd.DataFrame()
    if not tier_data.empty:
        drugs_list = ', '.join(
            f"{r['Treatment'].title()} ({r['% Positive']:.0f}% positive, n={r['Users']})"
            for _, r in tier_data.head(5).iterrows()
        )
        summary_parts.append(f'- **{tier_name} evidence:** {drugs_list}')

if causal_exclude:
    summary_parts.append(
        f'- **Excluded from recommendations:** {", ".join(c.title() for c in causal_exclude)} '
        '(negative sentiment reflects causal context, not treatment failure).'
    )

summary_parts.append('\n### Caveats\n')
summary_parts.append(
    '- **Self-reported community data** -- not equivalent to a clinical trial. '
    'Selection bias means users with strong opinions (positive or negative) are '
    'more likely to post.'
)
summary_parts.append(
    '- **1-month snapshot** -- treatment discussions may vary over time as new '
    'evidence and trends emerge.'
)
summary_parts.append(
    '- **Sample sizes** -- many treatments have fewer than 15 reporters. '
    'Statistical power is limited for small-n treatments.'
)
summary_parts.append(
    '- **Confounders** -- users often try multiple treatments simultaneously. '
    'We cannot attribute outcomes to a single treatment without controlled studies.'
)

summary_parts.append('\n### Suggested Next Steps\n')
summary_parts.append(
    '- Collect 3-6 months of data to increase per-treatment sample sizes.'
)
summary_parts.append(
    '- Run logistic regression to control for demographics and comorbidities.'
)
summary_parts.append(
    '- Cross-reference findings with published clinical trial data for the top-tier treatments.'
)
summary_parts.append(
    '- Stratify by comorbidity (fatigue + POTS vs fatigue + brain fog) to identify '
    'condition-specific treatment signals.'
)

summary_parts.append('\n---\n')
summary_parts.append(f'*{REPORTING_BIAS_DISCLAIMER}*')

display(Markdown('\n'.join(summary_parts)))

In [ ]:
conn.close()